In [74]:
using Gridap
using Gridap.FESpaces
using Gridap.ReferenceFEs
using Gridap.Arrays
using Gridap.Geometry
using Gridap.Fields
using Gridap.CellData
using FillArrays
using Test
using InteractiveUtils
using Gridap
using Gridap.Fields
using Gridap.Polynomials
using Gridap.ReferenceFEs
using LinearAlgebra
using FillArrays
using StaticArrays

In [ ]:
# -----------------------------
# Material parameters
# -----------------------------
E = 119e3
ν = 0.3
C = (E / (1 - ν^2)) * [ 1.0  ν   0.0;
                        ν    1.0 0.0;
                        0.0  0.0 (1-ν)/2 ]
# -----------------------------
# Mesh
# -----------------------------
L, W = 60.0, 20.0
partition = (Int(L*3), Int(W*3))
model = CartesianDiscreteModel((0.0,L,0.0,W), partition)

labels = get_face_labeling(model)
add_tag_from_tags!(labels, "left", [1,3,7])
add_tag_from_tags!(labels, "right", [2,4,8])
order = 1
reffe = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
V = FESpace(model,reffe)
U = V
Ω = Triangulation(model)
tria = Triangulation(model)
trian = get_triangulation(model)
ncells = num_cells(tria)
cell_coords = get_cell_coordinates(tria)

# -----------------------------
# Shape functions (bilinear)
# -----------------------------
ref_nodes = Point{2,Float64}[(0,0),(1,0),(0,1),(1,1)]
filter(e,p) = true
m = MonomialBasis{2}(Float64,1,filter)
l = LagrangianDofBasis(Float64,ref_nodes)
change = inv(evaluate(l,m))
s = linear_combination(change,m)

# Quadrature (1-point)
q = [Point{2,Float64}((0.5,0.5))]
w = [1.0]

cell_s = Fill(s, ncells)
cell_q = Fill(q, ncells)
cell_w = Fill(w, ncells)

cell_φ = lazy_map(linear_combination, cell_coords, cell_s)
cell_Jt = lazy_map(∇, cell_φ)
cell_detJ = lazy_map(Broadcasting(det), cell_Jt)
cell_invJt = lazy_map(Operation(inv), cell_Jt)
cell_∇ref_s = lazy_map(Broadcasting(∇), cell_s)
cell_∇s = lazy_map(Broadcasting(Operation(⋅)), cell_invJt, cell_∇ref_s)
# -----------------------------
# B-matrix
# -----------------------------
function B_matrix!(B,∇s_q,nshape)     
    for i in 1:nshape
        dNdx = ∇s_q[i]
        B[1,2i-1] = dNdx[1]
        B[2,2i]   = dNdx[2]
        B[3,2i-1] = dNdx[2]
        B[3,2i]   = dNdx[1]
    end
    return nothing
end
# # -----------------------------
function Ke_func(∇s_cell, detJ_cell, wq)
    ∇s_q = [evaluate(∇s_cell[i], q[1]) for i in 1:length(∇s_cell)]
    nshape = length(∇s_q)   
    B = zeros(3,2*nshape)
    B_matrix!(B,∇s_q,nshape)
    return B' * C * B * abs(detJ_cell(q[1])) * wq[1]
end
# -----------------------------
# Compute all element stiffness matrices using lazy_map
# -----------------------------
Ke_collection = lazy_map(Ke_func, cell_∇s, cell_detJ, cell_w);



3.2066023976153728e7

In [90]:
# ========
n_nodes = num_nodes(model)
ndofs = 2*n_nodes
K_global = zeros(ndofs, ndofs)
F_global = zeros(ndofs)
# -----------------------------
# Assemble global stiffness
# -----------------------------
conn = get_cell_node_ids(tria)
for (icell, Ke) in enumerate(Ke_collection)
    nodes = conn[icell]
    for i in 1:4, j in 1:4
        K_global[2*nodes[i]-1, 2*nodes[j]-1] += Ke[2*i-1, 2*j-1]
        K_global[2*nodes[i]-1, 2*nodes[j]]   += Ke[2*i-1, 2*j]
        K_global[2*nodes[i],   2*nodes[j]-1] += Ke[2*i,   2*j-1]
        K_global[2*nodes[i],   2*nodes[j]]   += Ke[2*i,   2*j]
    end
end
norm(K_global)

3.2066023976153728e7

In [91]:
using SparseArrays

# === assume everything above is already defined: model, tria, Ke_collection, etc. ===
n_nodes = num_nodes(model)
ndofs = 2 * n_nodes

# Connectivity: conn is a Vector of 4-element vectors of node ids
conn = get_cell_node_ids(tria)
nelems = length(conn)
ndofs_per_elem = 8          # 4 nodes * 2 dofs/node
entries_per_elem = ndofs_per_elem * ndofs_per_elem
total_entries = nelems * entries_per_elem

# Preallocate COO arrays
I = Vector{Int}(undef, total_entries)
J = Vector{Int}(undef, total_entries)
V = Vector{Float64}(undef, total_entries)

# Build element DOF map (matrix nelems x 8)
edofs = Matrix{Int}(undef, nelems, ndofs_per_elem)
for (icell, nodes) in enumerate(conn)
    # ordering: node1_x, node1_y, node2_x, node2_y, ...
    d = Int[]
    for ni in nodes
        push!(d, 2*ni - 1)
        push!(d, 2*ni)
    end
    edofs[icell, :] .= d
end

# Fill COO arrays
idx = 1
for (icell, Ke) in enumerate(Ke_collection)
    @inbounds dofs = edofs[icell, :]
    # Ke is 8×8 (2*4 × 2*4)
    @inbounds for a in 1:ndofs_per_elem
        ga = dofs[a]
        for b in 1:ndofs_per_elem
            gb = dofs[b]
            I[idx] = ga
            J[idx] = gb
            V[idx] = Ke[a,b]
            idx += 1
        end
    end
end

@assert idx - 1 == total_entries  # sanity check

# Build sparse global stiffness
K_global = sparse(I, J, V, ndofs, ndofs)

# (optional) force vector preallocated as before
F_global = zeros(ndofs)

# quick check
println("Assembled K_global: size = ", size(K_global), ", nnz = ", nnz(K_global))
println("||K|| = ", norm(K_global))


Assembled K_global: size = (22082, 22082), nnz = 391684
||K|| = 3.2066023976143878e7


In [ ]:
# -----------------------------
# Load vector (right edge)
# -----------------------------
Γ = BoundaryTriangulation(model, tags="right")
face_node_ids = get_cell_node_ids(Γ)
face_coords   = get_cell_coordinates(Γ)

# 
ref_nodes_line = Point{1,Float64}[Point(0.0), Point(1.0)]
m_line = MonomialBasis{1}(Float64,1)
l_line = LagrangianDofBasis(Float64, ref_nodes_line)
change_line = inv(evaluate(l_line,m_line))
s_line = linear_combination(change_line,m_line)
q_line = [Point{1,Float64}((0.5,))]
w_line = [1.0]
t = SVector(0.0, -1.0)

function Fe_func(s_line, q_line, w_line, t, coords)
    n_nodes = length(coords)          # must be 4
    @assert n_nodes == 4

    # Evaluate shape functions at quadrature point(s)
    N_vals = [evaluate(s_line[i], q_line[1]) for i in 1:n_nodes]

    # Edge length (Jacobian) – using nodes 1 and 2 of the quad face
    J = norm(coords[2] - coords[1])

    # Local force vector: 8 entries (2 DOFs per node)
    Fe = zeros(2*n_nodes)

    # x-contributions (odd indices)
    Fe[1:2:7] .= t[1] .* N_vals .* J .* w_line[1]

    # y-contributions (even indices)
    Fe[2:2:8] .= t[2] .* N_vals .* J .* w_line[1]

    return Fe
end
# -----------------------------
# Wrap inputs as lazy arrays
# -----------------------------
cell_s_line = lazy_map(identity, Fill(s_line, ncells))  # shape functions
cell_q_line = lazy_map(identity, Fill(q_line, ncells))  # quadrature points
cell_w_line = lazy_map(identity, Fill(w_line, ncells))  # quadrature weights
cell_t_line = lazy_map(identity, Fill(t, ncells))       # traction vector

# -----------------------------
# Face coordinates (boundary faces)
# -----------------------------
coords = get_cell_coordinates(Ω)  # Ω = boundary triangulation

# -----------------------------
# Compute all local face forces lazily
# -----------------------------
Fe_collection = lazy_map(Fe_func, cell_s_line, cell_q_line, cell_w_line, cell_t_line, coords);

# -----------------------------
# Inspect one element
# -----------------------------


10800-element LazyArray{Fill{typeof(Fe_func), 1, Tuple{Base.OneTo{Int64}}}, Vector{Float64}, 1, Tuple{Fill{Gridap.Fields.LinearCombinationFieldVector{Matrix{Float64}, MonomialBasis{1, Float64}}, 1, Tuple{Base.OneTo{Int64}}}, Fill{Vector{VectorValue{1, Float64}}, 1, Tuple{Base.OneTo{Int64}}}, Fill{Vector{Float64}, 1, Tuple{Base.OneTo{Int64}}}, Fill{SVector{2, Float64}, 1, Tuple{Base.OneTo{Int64}}}, LazyArray{Fill{Broadcasting{Reindex{Gridap.Geometry.CartesianCoordinates{2, Float64, typeof(identity)}}}, 2, Tuple{Base.OneTo{Int64}, Base.OneTo{Int64}}}, Vector{VectorValue{2, Float64}}, 2, Tuple{Gridap.Geometry.CartesianCellNodes{2}}}}}:
 [0.0, -0.16666666666666666, 0.0, -0.16666666666666666, 0.0, -1.586541845646e-312, 0.0, -3.173083691217e-312]
 [0.0, -0.16666666666666666, 0.0, -0.16666666666666666, 0.0, -1.586541845646e-312, 0.0, -3.173083691217e-312]
 [0.0, -0.16666666666666669, 0.0, -0.16666666666666669, 0.0, -1.586541845646e-312, 0.0, -3.173083691217e-312]
 [0.0, -0.16666666666666663, 

3.2066023976153728e7

In [ ]:
using SparseArrays
cell_mat = lazy_map(Ke_func, cell_∇s, cell_detJ, cell_w)
# Cell-wise dof ids
cell_dofs = get_cell_dof_ids(V)
# Assembly
assem = SparseMatrixAssembler(V,V)
data = ([cell_mat],[cell_dofs],[cell_dofs])
K_ff = assemble_matrix(assem,data)

# ========
cellvec = Fe_collection;
rows = get_cell_dof_ids(V)
assem = SparseMatrixAssembler(V,V)
vecdata = ([cellvec],[rows])
F_global =  assemble_vector(assem,vecdata)
# -----------------------------
# Apply Dirichlet BC (left edge fixed)
# -----------------------------
Γ_left = BoundaryTriangulation(model, tags="left")
left_nodes = unique(vcat(get_cell_node_ids(Γ_left)...))
fixed_dofs = vcat(2*left_nodes .- 1, 2*left_nodes)
free_dofs = setdiff(1:ndofs, fixed_dofs)
# ========
K_ff = K_global[free_dofs, free_dofs]
F_f  = F_global[free_dofs]
# -----------------------------
# Solve
# -----------------------------
U_low_API = zeros(ndofs)
U_low_API[free_dofs] = K_ff \ F_f;

print(norm(U_low_API))

3.4650534846223064

In [81]:

degree = 1
Ω = Triangulation(model)
dΩ = Measure(Ω, degree)
Γ = BoundaryTriangulation(model, tags="right")
dΓ = Measure(Γ, degree)
# -----------------------------
# FE space
# -----------------------------
order = 1
reffe = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
Vh = TestFESpace(Ω, reffe; conformity=:H1, dirichlet_tags="left")
Uh = TrialFESpace(Vh)
# -----------------------------
# Plane stress constitutive matrix (component-wise)
# -----------------------------
# 2D plane stress Lamé parameters
λ = E*ν/(1-ν^2)
μ = E/(2*(1+ν))
ε₀(u) = 0.5 * ( ∇(u) + transpose(∇(u))) # Strain; In Gridap this can be automatically defined in Gridap.ε
σ(ε₀) = λ * tr(ε₀) * one(ε₀) + 2 * μ * ε₀
# The weak form
a(u,v) = ∫((σ∘ε₀(u)) ⊙ ε₀(v))*dΩ # Left-hand size; (∘) Composite functions
Fvec(x) = VectorValue(0.0, -1.0)
l2(v) = ∫(Fvec ⋅ v) * dΓ
# -----------------------------
# Solve
# -----------------------------
op = AffineFEOperator(a, l2, Uh, Vh)
uh_high = solve(op)

SingleFieldFEFunction():
 num_cells: 10800
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 5707963658093428275

In [82]:
U_high_API = get_free_dof_values(uh_high) # displacement vector
println("Norm of uh in high API = ", norm(U_high_API), "\n")


Norm of uh in high API = 1.035825683671591

